Uploading the original excel file with all sheets

In [ ]:
# importing libraries
import pandas as pd
import numpy as np
from google.colab import drive

# Connecting Google Drive
drive.mount('/content/drive')

# google drive file path
file_path = '/content/drive/MyDrive/Netflix/NetflixAnalysis.xlsx'


Mounted at /content/drive


Transforming the Hours Sheet

In [ ]:
hrssheet = '2023 - 2026 hrs'

raw_hrs = pd.read_excel(file_path, sheet_name=hrssheet)
raw_hrs.head()

,Title Name,Runtime,Type,2023 H1 Hours,2023 H2 Hours,2024 H1 Hours,2024 H2 Hours,2025 H1 Hours,2025 H2 Hours,2026 H1 Hours,Total Hours Viewed
0,Squid Game: Season 2,07:10:00,TV,-,-,-,619900000,840300000,122400000,32000000,1614600000
1,The Night Agent: Season 1,08:11:00,TV,812100000,155500000,132800000,83000000,213800000,48100000,109800000,1555100000
2,Stranger Things 5,10:24:00,TV,-,-,-,-,-,879700000,578400000,1458100000
3,Stranger Things 4,13:04:00,TV,133600000,97100000,110100000,109900000,135500000,562200000,303300000,1451700000
4,Wednesday: Season 1,06:49:00,TV,507700000,162700000,104200000,118800000,90400000,320100000,51500000,1355400000


In [ ]:
# adding the period to rows instead of columns
hrs = raw_hrs.melt(
    id_vars=["Title Name", "Runtime", "Type"],
    value_vars=[
        "2023 H1 Hours",
        "2023 H2 Hours",
        "2024 H1 Hours",
        "2024 H2 Hours",
        "2025 H1 Hours",
        "2025 H2 Hours"
    ],
    var_name="Period",
    value_name="HoursViewed"
)
hrs["Period"] = hrs["Period"].str.replace(" Hours", "", regex=False)
hrs

,Title Name,Runtime,Type,Period,HoursViewed
0,Squid Game: Season 2,07:10:00,TV,2023 H1,-
1,The Night Agent: Season 1,08:11:00,TV,2023 H1,812100000
2,Stranger Things 5,10:24:00,TV,2023 H1,-
3,Stranger Things 4,13:04:00,TV,2023 H1,133600000
4,Wednesday: Season 1,06:49:00,TV,2023 H1,507700000
...,...,...,...,...,...
195451,Wrong Turn at Tahoe,01:31:00,Movie,2025 H2,-
195452,Yesterday Today Tomorrow (2011),01:56:00,Movie,2025 H2,-
195453,You Are on My Mind,01:49:00,Movie,2025 H2,-
195454,¬°Ay mi madre!,01:21:00,Movie,2025 H2,-


In [ ]:
# converting the runtime into minutes
hrs[["Year", "Half"]] = hrs["Period"].str.extract(
    r"(\d{4}) (H[12])"
)

hrs["Year"] = hrs["Year"].astype(int)
hrs["Runtime_original"] = hrs["Runtime"]

hrs["Runtime"] = pd.to_timedelta(
    hrs["Runtime"].astype(str),  # Convert to string to handle datetime.time objects
    errors="coerce"
)

hrs["Runtime_minutes"] = hrs["Runtime"].dt.total_seconds() / 60


In [ ]:
# taking out all - in the cells
hrs["Released"] = hrs["HoursViewed"].ne("-")

hrs["HoursViewed"] = hrs["HoursViewed"].replace("-", np.nan)

hrs

/tmp/ipykernel_940/174463492.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  hrs["HoursViewed"] = hrs["HoursViewed"].replace("-", np.nan)


,Title Name,Runtime,Type,Period,HoursViewed,Year,Half,Runtime_original,Runtime_minutes,Released
0,Squid Game: Season 2,0 days 07:10:00,TV,2023 H1,NaN,2023,H1,07:10:00,430.0,False
1,The Night Agent: Season 1,0 days 08:11:00,TV,2023 H1,812100000.0,2023,H1,08:11:00,491.0,True
2,Stranger Things 5,0 days 10:24:00,TV,2023 H1,NaN,2023,H1,10:24:00,624.0,False
3,Stranger Things 4,0 days 13:04:00,TV,2023 H1,133600000.0,2023,H1,13:04:00,784.0,True
4,Wednesday: Season 1,0 days 06:49:00,TV,2023 H1,507700000.0,2023,H1,06:49:00,409.0,True
...,...,...,...,...,...,...,...,...,...,...
195451,Wrong Turn at Tahoe,0 days 01:31:00,Movie,2025 H2,NaN,2025,H2,01:31:00,91.0,False
195452,Yesterday Today Tomorrow (2011),0 days 01:56:00,Movie,2025 H2,NaN,2025,H2,01:56:00,116.0,False
195453,You Are on My Mind,0 days 01:49:00,Movie,2025 H2,NaN,2025,H2,01:49:00,109.0,False
195454,¬°Ay mi madre!,0 days 01:21:00,Movie,2025 H2,NaN,2025,H2,01:21:00,81.0,False


In [ ]:
hrs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195456 entries, 0 to 195455
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype          
---  ------            --------------   -----          
 0   Title Name        195456 non-null  object         
 1   Runtime           170028 non-null  timedelta64[ns]
 2   Type              195456 non-null  object         
 3   Period            195456 non-null  object         
 4   HoursViewed       98264 non-null   float64        
 5   Year              195456 non-null  int64          
 6   Half              195456 non-null  object         
 7   Runtime_original  172806 non-null  object         
 8   Runtime_minutes   170028 non-null  float64        
 9   Released          195456 non-null  bool           
dtypes: bool(1), float64(2), int64(1), object(5), timedelta64[ns](1)
memory usage: 13.6+ MB


In [ ]:
# checking data types
hrs["Title Name"].map(type).value_counts()

,count
Title Name,
<class 'str'>,195258
<class 'int'>,126
<class 'datetime.datetime'>,66
<class 'datetime.time'>,6


In [ ]:
# viewing all non string title names to try to convert into a string
non_string_titles_hrs = hrs[
    ~hrs["Title Name"].map(lambda x: isinstance(x, str))
][["Title Name", "Type", "Runtime_original"]]

non_string_titles_hrs.drop_duplicates()

,Title Name,Type,Runtime_original
1871,65,Movie,01:33:00
2219,2012,Movie,02:38:00
2714,180,Movie,01:35:00
4635,1922,Movie,01:43:00
5076,2026-07-22 00:00:00,Movie,02:24:00
5093,1917,Movie,01:59:00
5704,300,Movie,01:57:00
5878,1408,Movie,01:44:00
8799,404,Movie,01:43:00
10117,211,Movie,01:27:00


In [ ]:
# making integer columns into names because python automatically detected a number
import datetime

def clean_titleh(x):
    if isinstance(x, int):
        return str(x)
    return x

hrs["Title Name"] = hrs["Title Name"].apply(clean_titleh)

In [ ]:
# displaying and title names stored as a datetime to try and find the real name
date_problemh = hrs[
    hrs["Title Name"].map(
        lambda x: isinstance(
            x,
            (datetime.datetime, datetime.time)
        )
    )
]

date_problemh[
    ["Title Name", "Type", "Runtime_original"]
].drop_duplicates()

,Title Name,Type,Runtime_original
5076,2026-07-22 00:00:00,Movie,02:24:00
17494,2045-06-01 00:00:00,Movie,01:53:00
19177,2026-09-05 00:00:00,Movie,01:31:00
25515,2026-05-18 00:00:00,Movie,02:00:00
28791,2026-07-24 00:00:00,Movie,01:22:00
29607,11:11:00,Movie,NaN
29610,2026-08-15 00:00:00,Movie,NaN
30723,2026-10-01 00:00:00,Movie,NaN


In [ ]:
# re-checking data types
hrs["Title Name"].map(type).value_counts()

,count
Title Name,
<class 'str'>,195384
<class 'datetime.datetime'>,66
<class 'datetime.time'>,6


In [ ]:
# manually went and found all title names imported as datetime
title_fixes = {
    pd.Timestamp("2026-07-22"): "22 July",
    pd.Timestamp("2045-06-01"): "6/45",
    pd.Timestamp("2026-09-05"): "September 5",
    pd.Timestamp("2026-05-18"): "May 18",
    pd.Timestamp("2026-07-24"): "7-24",
    pd.Timestamp("2026-08-15"): "15 August",
    pd.Timestamp("2026-10-01"): "October 1",
}

hrs["Title Name"] = hrs["Title Name"].replace(title_fixes)

hrs["Title Name"] = hrs["Title Name"].replace({
    datetime.time(11, 11): "11:11"
})

In [ ]:
#grouping the columns to calcuate the total hours viewed and putting it
#into the cleaned dataframe

hrs_clean = (
    hrs
    .groupby(
        ["Title Name", "Type", "Runtime_minutes", "Period", "Year", "Half"],
        dropna=False,
        as_index=False
    )
    .agg(
        HoursViewed=(
            "HoursViewed",
            lambda x: x.sum(min_count=1)
        )
    )
)

In [ ]:
# created a released column to denote if a title has been released
hrs_clean["Released"] = hrs_clean["HoursViewed"].notna()

In [ ]:
# checking the row count for all titles
# should be 6 for each half of 3 years
hrs_clean.groupby("Title Name").size().sort_values(ascending=False).head(20)

,0
Title Name,
◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,6
"""Sr.""",6
#Alive,6
#AnneFrank - Parallel Stories,6
#AtFirstSight,6
#FriendButMarried,6
#FriendButMarried 2,6
#Iamhere,6
#LadyRancho,6


In [ ]:
# final cleaned table
hrs_clean

,Title Name,Type,Runtime_minutes,Period,Year,Half,HoursViewed,Released
0,"""Sr.""",Movie,89.0,2023 H1,2023,H1,NaN,False
1,"""Sr.""",Movie,89.0,2023 H2,2023,H2,300000.0,True
2,"""Sr.""",Movie,89.0,2024 H1,2024,H1,100000.0,True
3,"""Sr.""",Movie,89.0,2024 H2,2024,H2,NaN,False
4,"""Sr.""",Movie,89.0,2025 H1,2025,H1,NaN,False
...,...,...,...,...,...,...,...,...
195427,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2023 H2,2023,H2,NaN,False
195428,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2024 H1,2024,H1,NaN,False
195429,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2024 H2,2024,H2,NaN,False
195430,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2025 H1,2025,H1,NaN,False


saving to google drive

In [ ]:
# uplaoding the cleaned file to google drive
file_path = '/content/drive/MyDrive/Netflix/EngagementHoursCleaned.csv'
hrs_clean.to_csv(file_path, index=False)

#Transforming the Views sheet

In [ ]:
# acessing the raw views sheet
viewssheet = '2023 - 2026 views'

raw_views = pd.read_excel(file_path, sheet_name=viewssheet)
raw_views.head()

,Title Name,Runtime,Type,2023 H1 Views,2023 H2 Views,2024 H1 Views,2024 H2 Views,2025 H1 Views,2025 H2 Views,2026 H1 Views,Total Views
0,KPop Demon Hunters,01:40:00,Movie,-,-,-,-,36700000,481600000,130400000,648700000
1,The Boss Baby,01:38:00,Movie,46000000,61700000,63600000,40800000,55200000,18900000,25300000,311500000
2,Sing (2016),01:48:00,Movie,44700000,40800000,33600000,58200000,41100000,44100000,19300000,281800000
3,The Super Mario Bros. Movie,01:32:00,Movie,-,44900000,80300000,45800000,38900000,31300000,30500000,271700000
4,Shrek,01:30:00,Movie,32100000,35100000,56900000,8800000,53500000,41700000,24300000,252400000


In [ ]:
# transforming the period into rows
views = raw_views.melt(
    id_vars=["Title Name", "Runtime", "Type"],
    value_vars=[
        "2023 H1 Views",
        "2023 H2 Views",
        "2024 H1 Views",
        "2024 H2 Views",
        "2025 H1 Views",
        "2025 H2 Views"
    ],
    var_name="Period",
    value_name="HoursViewed"
)
views["Period"] = views["Period"].str.replace(" Views", "", regex=False)
views

,Title Name,Runtime,Type,Period,HoursViewed
0,KPop Demon Hunters,01:40:00,Movie,2023 H1,-
1,The Boss Baby,01:38:00,Movie,2023 H1,46000000
2,Sing (2016),01:48:00,Movie,2023 H1,44700000
3,The Super Mario Bros. Movie,01:32:00,Movie,2023 H1,-
4,Shrek,01:30:00,Movie,2023 H1,32100000
...,...,...,...,...,...
195451,ÿßŸÑŸÜÿßŸÖŸàÿ≥: Season 1,NaN,TV,2025 H2,-
195452,ÿ≠ŸÉÿßŸäÿßÿ™ ÿ®ŸÜÿßÿ™ ÿßŸÑÿ¨ÿ≤ÿ° Ÿ°: Season 2,NaN,TV,2025 H2,-
195453,ÿ≠ŸÉÿßŸäÿßÿ™ ÿ®ŸÜÿßÿ™ ÿßŸÑÿ¨ÿ≤ÿ° Ÿ°: Season 3,NaN,TV,2025 H2,-
195454,ÿ±ÿßÿ≥ ÿßŸÑÿ≥ŸÜÿ©,NaN,Movie,2025 H2,-


In [ ]:
# extracting H1 and H2 periods
views[["Year", "Half"]] = views["Period"].str.extract(
    r"(\d{4}) (H[12])"
)

views["Year"] = views["Year"].astype(int)

# calculating runtime minutes
views["Runtime_original"] = views["Runtime"]

views["Runtime"] = pd.to_timedelta(
    views["Runtime"].astype(str),  # Convert to string to handle datetime.time objects
    errors="coerce"
)

views["Runtime_minutes"] = views["Runtime"].dt.total_seconds() / 60


In [ ]:
# taking out all - in the cells
views["Released"] = views["HoursViewed"].ne("-")

views["HoursViewed"] = views["HoursViewed"].replace("-", np.nan)

views

/tmp/ipykernel_940/79970989.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  views["HoursViewed"] = views["HoursViewed"].replace("-", np.nan)


,Title Name,Runtime,Type,Period,HoursViewed,Year,Half,Runtime_original,Runtime_minutes,Released
0,KPop Demon Hunters,0 days 01:40:00,Movie,2023 H1,NaN,2023,H1,01:40:00,100.0,False
1,The Boss Baby,0 days 01:38:00,Movie,2023 H1,46000000.0,2023,H1,01:38:00,98.0,True
2,Sing (2016),0 days 01:48:00,Movie,2023 H1,44700000.0,2023,H1,01:48:00,108.0,True
3,The Super Mario Bros. Movie,0 days 01:32:00,Movie,2023 H1,NaN,2023,H1,01:32:00,92.0,False
4,Shrek,0 days 01:30:00,Movie,2023 H1,32100000.0,2023,H1,01:30:00,90.0,True
...,...,...,...,...,...,...,...,...,...,...
195451,ÿßŸÑŸÜÿßŸÖŸàÿ≥: Season 1,NaT,TV,2025 H2,NaN,2025,H2,NaN,NaN,False
195452,ÿ≠ŸÉÿßŸäÿßÿ™ ÿ®ŸÜÿßÿ™ ÿßŸÑÿ¨ÿ≤ÿ° Ÿ°: Season 2,NaT,TV,2025 H2,NaN,2025,H2,NaN,NaN,False
195453,ÿ≠ŸÉÿßŸäÿßÿ™ ÿ®ŸÜÿßÿ™ ÿßŸÑÿ¨ÿ≤ÿ° Ÿ°: Season 3,NaT,TV,2025 H2,NaN,2025,H2,NaN,NaN,False
195454,ÿ±ÿßÿ≥ ÿßŸÑÿ≥ŸÜÿ©,NaT,Movie,2025 H2,NaN,2025,H2,NaN,NaN,False


In [ ]:
views.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195456 entries, 0 to 195455
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype          
---  ------            --------------   -----          
 0   Title Name        195456 non-null  object         
 1   Runtime           170028 non-null  timedelta64[ns]
 2   Type              195456 non-null  object         
 3   Period            195456 non-null  object         
 4   HoursViewed       93452 non-null   float64        
 5   Year              195456 non-null  int64          
 6   Half              195456 non-null  object         
 7   Runtime_original  172806 non-null  object         
 8   Runtime_minutes   170028 non-null  float64        
 9   Released          195456 non-null  bool           
dtypes: bool(1), float64(2), int64(1), object(5), timedelta64[ns](1)
memory usage: 13.6+ MB


In [ ]:
# checking data types of title names
views["Title Name"].map(type).value_counts()

,count
Title Name,
<class 'str'>,195258
<class 'int'>,126
<class 'datetime.datetime'>,66
<class 'datetime.time'>,6


In [ ]:
# viewing non string title names
non_string_titles = views[
    ~views["Title Name"].map(lambda x: isinstance(x, str))
][["Title Name", "Type", "Runtime_original"]]

non_string_titles.drop_duplicates()

,Title Name,Type,Runtime_original
453,65,Movie,01:33:00
861,180,Movie,01:35:00
1363,2012,Movie,02:38:00
2243,1922,Movie,01:43:00
3047,1917,Movie,01:59:00
3288,1408,Movie,01:44:00
3488,300,Movie,01:57:00
4669,2026-07-22 00:00:00,Movie,02:24:00
5921,404,Movie,01:43:00
6379,211,Movie,01:27:00


In [ ]:
# converts integer names to strings
import datetime

def clean_title(x):
    if isinstance(x, int):
        return str(x)
    return x

views["Title Name"] = views["Title Name"].apply(clean_title)

In [ ]:
# views date time title names
date_problem = views[
    views["Title Name"].map(
        lambda x: isinstance(
            x,
            (datetime.datetime, datetime.time)
        )
    )
]

date_problem[
    ["Title Name", "Type", "Runtime_original"]
].drop_duplicates()

,Title Name,Type,Runtime_original
4669,2026-07-22 00:00:00,Movie,02:24:00
14908,2045-06-01 00:00:00,Movie,01:53:00
15633,2026-09-05 00:00:00,Movie,01:31:00
22705,2026-05-18 00:00:00,Movie,02:00:00
25224,2026-07-24 00:00:00,Movie,01:22:00
30656,11:11:00,Movie,NaN
30659,2026-08-15 00:00:00,Movie,NaN
31773,2026-10-01 00:00:00,Movie,NaN


In [ ]:
# cheecking data types again
views["Title Name"].map(type).value_counts()

,count
Title Name,
<class 'str'>,195384
<class 'datetime.datetime'>,66
<class 'datetime.time'>,6


In [ ]:
# views datetime title names
date_titles = views[
    views["Title Name"].map(
        lambda x: isinstance(x, (datetime.datetime, datetime.time))
    )
][["Title Name", "Type", "Runtime_original"]].drop_duplicates()

date_titles

,Title Name,Type,Runtime_original
4669,2026-07-22 00:00:00,Movie,02:24:00
14908,2045-06-01 00:00:00,Movie,01:53:00
15633,2026-09-05 00:00:00,Movie,01:31:00
22705,2026-05-18 00:00:00,Movie,02:00:00
25224,2026-07-24 00:00:00,Movie,01:22:00
30656,11:11:00,Movie,NaN
30659,2026-08-15 00:00:00,Movie,NaN
31773,2026-10-01 00:00:00,Movie,NaN


In [ ]:
# same list as before was manually checked
title_fixes = {
    pd.Timestamp("2026-07-22"): "22 July",
    pd.Timestamp("2045-06-01"): "6/45",
    pd.Timestamp("2026-09-05"): "September 5",
    pd.Timestamp("2026-05-18"): "May 18",
    pd.Timestamp("2026-07-24"): "7-24",
    pd.Timestamp("2026-08-15"): "15 August",
    pd.Timestamp("2026-10-01"): "October 1",
}

views["Title Name"] = views["Title Name"].replace(title_fixes)

views["Title Name"] = views["Title Name"].replace({
    datetime.time(11, 11): "11:11"
})

In [ ]:
# final data type check
views["Title Name"].map(type).value_counts()

,count
Title Name,
<class 'str'>,195456


In [ ]:
# checking number of unique title names
views["Title Name"].nunique()

32572

In [ ]:
# checking the row count for all titles
# should be 6 for each half of 3 years, but some have 12
views.groupby("Title Name", sort=False).size().sort_values(ascending=False).head(20)

,0
Title Name,
7-24,12
6/45,12
May 18,12
22 July,12
The Liar and His Lover,6
Evil (2019): Season 1,6
My Turn: 2025,6
Rondo Behind the Final Curtain: Season 1,6
Drake and Josh: Season 3,6


In [ ]:
# looking at examples with 12 entries
views[
    views["Title Name"].isin(["7-14", "6/45", "May 18", "22 July"])
].sort_values(["Title Name", "Period"])

,Title Name,Runtime,Type,Period,HoursViewed,Year,Half,Runtime_original,Runtime_minutes,Released
4669,22 July,0 days 02:24:00,Movie,2023 H1,NaN,2023,H1,02:24:00,144.0,False
9499,22 July,0 days 02:24:00,Movie,2023 H1,NaN,2023,H1,02:24:00,144.0,False
37245,22 July,0 days 02:24:00,Movie,2023 H2,NaN,2023,H2,02:24:00,144.0,False
42075,22 July,0 days 02:24:00,Movie,2023 H2,2900000.0,2023,H2,02:24:00,144.0,True
69821,22 July,0 days 02:24:00,Movie,2024 H1,2700000.0,2024,H1,02:24:00,144.0,True
74651,22 July,0 days 02:24:00,Movie,2024 H1,NaN,2024,H1,02:24:00,144.0,False
102397,22 July,0 days 02:24:00,Movie,2024 H2,1600000.0,2024,H2,02:24:00,144.0,True
107227,22 July,0 days 02:24:00,Movie,2024 H2,NaN,2024,H2,02:24:00,144.0,False
134973,22 July,0 days 02:24:00,Movie,2025 H1,1500000.0,2025,H1,02:24:00,144.0,True
139803,22 July,0 days 02:24:00,Movie,2025 H1,NaN,2025,H1,02:24:00,144.0,False


In [ ]:
# grouping the columns to calcuate the total views and putting it
# into the cleaned dataframe
views_clean = (
    views
    .groupby(
        ["Title Name", "Type", "Runtime_minutes", "Period", "Year", "Half"],
        dropna=False,
        as_index=False
    )
    .agg(
        HoursViewed=(
            "HoursViewed",
            lambda x: x.sum(min_count=1)
        )
    )
)

In [ ]:
# creating a released column to note whether engagement was recorded at the time
views_clean["Released"] = views_clean["HoursViewed"].notna()

In [ ]:
# checking counts again
views_clean.groupby("Title Name").size().sort_values(ascending=False).head(20)

,0
Title Name,
◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,6
"""Sr.""",6
#Alive,6
#AnneFrank - Parallel Stories,6
#AtFirstSight,6
#FriendButMarried,6
#FriendButMarried 2,6
#Iamhere,6
#LadyRancho,6


In [ ]:
# final cleaned data
views_clean

,Title Name,Type,Runtime_minutes,Period,Year,Half,HoursViewed,Released
0,"""Sr.""",Movie,89.0,2023 H1,2023,H1,NaN,False
1,"""Sr.""",Movie,89.0,2023 H2,2023,H2,200000.0,True
2,"""Sr.""",Movie,89.0,2024 H1,2024,H1,100000.0,True
3,"""Sr.""",Movie,89.0,2024 H2,2024,H2,NaN,False
4,"""Sr.""",Movie,89.0,2025 H1,2025,H1,NaN,False
...,...,...,...,...,...,...,...,...
195427,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2023 H2,2023,H2,NaN,False
195428,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2024 H1,2024,H1,NaN,False
195429,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2024 H2,2024,H2,NaN,False
195430,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2025 H1,2025,H1,NaN,False


In [ ]:
# adding to google drive
file_path = '/content/drive/MyDrive/Netflix/EngagementViewsCleaned.csv'
views_clean.to_csv(file_path, index=False)

# Combining the Files


In [ ]:
# reads the 2 cleaned datasets
hours = pd.read_csv('/content/drive/MyDrive/Netflix/EngagementHoursCleaned.csv')
views = pd.read_csv('/content/drive/MyDrive/Netflix/EngagementViewsCleaned.csv')

hours = hours.rename(
    columns={"HoursViewed": "HoursViewed"}
)

views = views.rename(
    columns={"HoursViewed": "Views"} # mistakenly named hours viewed
)

In [ ]:
# merging the 2 datasets into 1 called engagement
engagement = hours.merge(
    views[
        ["Title Name", "Type", "Runtime_minutes",
         "Period", "Views"]
    ],
    on=[
        "Title Name",
        "Type",
        "Runtime_minutes",
        "Period"
    ],
    how="outer",
    validate="one_to_one"
)

In [ ]:
# creates calculated vuews by using hours viewed and runtime
engagement["CalculatedViews"] = (
    engagement["HoursViewed"] /
    (engagement["Runtime_minutes"] / 60)
)

In [ ]:
# displays the new dataframe
engagement[
    ["Title Name", "Type", "Runtime_minutes", "Period",
     "HoursViewed", "Views", "CalculatedViews", "Released"]
].head(20)

,Title Name,Type,Runtime_minutes,Period,HoursViewed,Views,CalculatedViews,Released
0,"""Sr.""",Movie,89.0,2023 H1,NaN,NaN,NaN,False
1,"""Sr.""",Movie,89.0,2023 H2,300000.0,200000.0,2.022472e+05,True
2,"""Sr.""",Movie,89.0,2024 H1,100000.0,100000.0,6.741573e+04,True
3,"""Sr.""",Movie,89.0,2024 H2,NaN,NaN,NaN,False
4,"""Sr.""",Movie,89.0,2025 H1,NaN,NaN,NaN,False
5,"""Sr.""",Movie,89.0,2025 H2,NaN,NaN,NaN,False
6,#Alive,Movie,98.0,2023 H1,10700000.0,6600000.0,6.551020e+06,True
7,#Alive,Movie,98.0,2023 H2,7500000.0,4600000.0,4.591837e+06,True
8,#Alive,Movie,98.0,2024 H1,7100000.0,4300000.0,4.346939e+06,True
9,#Alive,Movie,98.0,2024 H2,6000000.0,3700000.0,3.673469e+06,True


In [ ]:
# creates column denoting if there was data reported
engagement["EngagementReported"] = (
    engagement["HoursViewed"].notna()
    | engagement["Views"].notna()
)

In [ ]:
# dropping released because we dont want to assume that's the reason
engagement = engagement.drop(columns=['Released'])

In [ ]:
# final cleaned dataframe
engagement

,Title Name,Type,Runtime_minutes,Period,Year,Half,HoursViewed,Views,CalculatedViews,EngagementReported
0,"""Sr.""",Movie,89.0,2023 H1,2023,H1,NaN,NaN,NaN,False
1,"""Sr.""",Movie,89.0,2023 H2,2023,H2,300000.0,200000.0,202247.191011,True
2,"""Sr.""",Movie,89.0,2024 H1,2024,H1,100000.0,100000.0,67415.730337,True
3,"""Sr.""",Movie,89.0,2024 H2,2024,H2,NaN,NaN,NaN,False
4,"""Sr.""",Movie,89.0,2025 H1,2025,H1,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...
195427,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2023 H2,2023,H2,NaN,NaN,NaN,False
195428,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2024 H1,2024,H1,NaN,NaN,NaN,False
195429,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2024 H2,2024,H2,NaN,NaN,NaN,False
195430,◊†◊ì◊ú◊¥◊ü-◊°◊ô◊§◊ï◊® ◊ê◊î◊ë◊î,Movie,99.0,2025 H1,2025,H1,NaN,NaN,NaN,False


In [ ]:
# uploading to google drive
file_path = '/content/drive/MyDrive/Netflix/Engagement.csv'
engagement.to_csv(file_path, index=False)